# Ders 13 - Cognee Bilgi Grafikleri ile Ajan Hafızası


## Kurulum

Bu not defteri, [**Cognee**](https://www.cognee.ai/) bilgi grafiklerini ve **Microsoft Agent Framework** (MAF) kullanarak kalıcı belleğe sahip zeki bir **kodlama asistanı** nasıl oluşturulacağını göstermektedir.

Cognee, yapılandırılmamış metni, vektör gömme destekli yapılandırılmış, sorgulanabilir bir bilgi grafiğine dönüştürerek — ajanınıza zengin, ilişki farkında uzun vadeli bir hafıza sağlar.

### Öğrenecekleriniz
1. **Bilgi Grafiklerini Oluşturma**: Geliştirici profillerini ve en iyi uygulamaları yapılandırılmış, sorgulanabilir bilgiye dönüştürme.
2. **Cognee ile MAF Entegrasyonu**: Bir MAF ajanının Cognee’nin bilgi grafiğini sorgulamasına izin vermek için `@tool` işlevlerini kullanma.
3. **Oturum Farkındalıklı Konuşmalar**: Aynı oturumdaki birden fazla soru arasında bağlamı koruma.
4. **Uzun Vadeli Bellek**: Önemli bilgileri oturumlar arasında saklama ve yeni konuşmalarda geri getirme.

### Önkoşullar
- Python 3.9+
- Oturum yönetimi için yerel çalışan Redis (`docker run -d -p 6379:6379 redis`)
- Bir LLM API anahtarı (ör. OpenAI) — `.env` içinde `LLM_API_KEY` ayarlayın
- `.env` içinde `CACHING=true` (Cognee oturumları için gerekli)
- Dağıtılmış sohbet modeli olan bir Azure AI Foundry projesi
- `.env` içinde `AZURE_AI_PROJECT_ENDPOINT` ve `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI ile kimlik doğrulaması (`az login`) yapılmış


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Ajan Hafıza Türleri

Bu not defteri, ana Ders 13 not defterindeki aynı üç hafıza türünü inceler, ancak uzun süreli hafıza arka ucu olarak Cognee'yi kullanır:

| Hafıza Türü | Mekanizma | Ömür |
|---|---|---|
| **Çalışma** | `agent.create_session()` (MAF) | Tek konuşma |
| **Kısa süreli** | Cognee oturum önbelleği (Redis) | Tek oturum |
| **Uzun süreli** | Cognee bilgi grafiği + vektörler | Kalıcı |

### Cognee'nin Hafıza Mimarisi
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee Depolamayı Hazırlayın


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Bölüm 1 — Bilgi Tabanı Oluşturma

Kodlama asistanımız için kapsamlı bir bilgi tabanı oluşturmak üzere üç tür veri alıyoruz:

1. **Geliştirici Profili** — kişisel uzmanlık ve teknik geçmiş  
2. **Python En İyi Uygulamaları** — pratik yönergelerle Python’un Zen’i  
3. **Geçmiş Konuşmalar** — geliştiriciler ve yapay zeka asistanları arasındaki önceki soru-cevap oturumları


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Bilgi Grafiğini Görselleştirme

Cognee, çıkardığı varlıkların ve ilişkilerin etkileşimli bir HTML görselleştirmesini oluşturabilir.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Belleği Memify ile Zenginleştir

`memify()` bilgi grafiğini analiz eder ve kalıpları, en iyi uygulamaları ve kavramlar arasındaki ilişkileri tanımlayarak akıllı kurallar oluşturur.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Bölüm 2 — Cognee Araçlarıyla MAF Ajanı

Şimdi, `@tool` fonksiyonları aracılığıyla Cognee'nin bilgi grafiğini sorgulayabilen bir MAF ajanı oluşturuyoruz. Bu, ajanın oturumlar aracılığıyla konuşma bağlamını korurken grafik tabanlı anlamsal aramanın tam gücünden yararlanmasını sağlar.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Oturumlarla Çalışma Belleği

`AgentSession` ( `agent.create_session()` ile oluşturulur) bir oturum içinde çalışma belleği sağlar. Ajan, Cognee'nin uzun vadeli bilgi grafiğine sorgu yaparken önceki mesajlara da başvurabilir.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Yeni Oturum — Uzun Vadeli Bellek Kalıcıdır

Yeni bir oturuma başlamak çalışma belleğini temizler, ancak Cognee bilgi grafiği hâlâ kullanılabilir durumdadır. Temsilci, tamamen yeni bir sohbette aynı uzun vadeli bilgiyi alabilir.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Özet

Bu not defterinde **MAF'ın çalışma belleği** (`agent.create_session()`) ile **Cognee'nin uzun vadeli bilgi grafiğini** birleştiren bir kodlama asistanı oluşturdunuz.

### Öğrendikleriniz
1. **Bilgi grafiği oluşturma**: Cognee yapılandırılmamış metni alır ve bir grafik + vektör belleği oluşturur.
2. **memify ile grafiğin zenginleştirilmesi**: Mevcut grafiğiniz üzerine türetilmiş gerçekler ve daha zengin ilişkiler ekler.
3. **MAF + Cognee entegrasyonu**: `@tool` fonksiyonları, bir MAF ajanının Cognee'nin grafiğini doğal şekilde sorgulamasını sağlar.
4. **Çalışma belleği + uzun vadeli bellek**: `AgentSession` ( `agent.create_session()` aracılığıyla) oturum bağlamı sağlarken Cognee kalıcı bilgiyi sağlar.
5. **NodeSets ile filtreli arama**: Bilgi grafiğinin belirli alt kümelerine hedeflenmiş sorgular yapma (örneğin sadece prensipler).

### Temel Çıkarımlar
- **Cognee** ham metni yapılandırılmış, ilişki farkındalığı olan bir belleğe dönüştürür — düz bir vektör deposundan daha güçlüdür.
- **`@tool` fonksiyonları** MAF ajanları ile dış bilgi sistemleri arasında temiz bir köprü oluşturur.
- **`AgentSession`** ( `agent.create_session()` aracılığıyla) her konuşma bağlamını uzun ömürlü bilgiden ayrı tutar.
- Aynı bilgi grafiği birden fazla oturum ve ajan tarafından kullanılır.

### Gerçek Dünya Uygulamaları
- **Geliştirici yardımcıları**: Kod inceleme, olay analizi, mimari asistanları
- **Müşteri destek yardımcıları**: Ürün belgeleri, SSS ve CRM notları üzerinde destek ajanları
- **İç uzman yardımcıları**: Politikalar, hukuki veya güvenlik asistanları kılavuzlar üzerinden çıkarım yapar
- **Birleşik veri katmanları**: Yapılandırılmış ve yapılandırılmamış verileri sorgulanabilir tek bir grafik halinde birleştirme

### Sonraki Adımlar
- Cognee'de zamansal farkındalıkla denemeler yapmak
- Alan-spesifik grafik kalitesi için OWL ontolojisi tanımlamak
- Zaman içinde geri bildirim döngüleri ekleyerek sorgulamayı geliştirmek
- Aynı Cognee bellek katmanını paylaşan çoklu ajan sistemlerine ölçeklemek


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri servisi [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba gösterilmekle birlikte, otomatik çevirilerin hata veya yanlışlıklar içerebileceğini lütfen unutmayınız. Orijinal belge, kendi ana dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucunda ortaya çıkabilecek yanlış anlamalar veya yorumlamalar nedeniyle sorumluluk kabul edilmemektedir.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
